# TP6 — Detección de Postura Corporal con YOLO Pose
**Instituto de Formación Técnica Superior N° 33**  
**Materia:** Técnicas de Procesamiento de Imágenes

---

YOLO Pose es una variante de YOLOv8 que detecta **17 puntos clave del cuerpo humano** (keypoints) en tiempo real. Permite analizar posturas, movimientos y posiciones del cuerpo a partir de imágenes o video.

## Paso 1 — Instalar dependencias

In [ ]:
!pip install ultralytics -q
print('✅ Instalación completa')

## Paso 2 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

## Paso 3 — Cargar y mostrar la imagen

In [ ]:
import cv2
import matplotlib.pyplot as plt

ruta = '/content/drive/MyDrive/Procesamiento-Imagenes/TP6/postura.jpg'

img = cv2.imread(ruta)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))
plt.imshow(img_rgb)
plt.title('Imagen original — Buena y mala postura')
plt.axis('off')
plt.show()

## Paso 4 — Detección de puntos corporales con YOLO Pose

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

# Cargar modelo YOLO Pose
model = YOLO('yolov8n-pose.pt')

# Detectar puntos corporales
results = model(ruta)

# Mostrar resultado
img_resultado = results[0].plot()
img_resultado_rgb = cv2.cvtColor(img_resultado, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))
plt.imshow(img_resultado_rgb)
plt.title('Detección YOLO Pose — 17 puntos corporales')
plt.axis('off')
plt.show()

## Paso 5 — Ver puntos detectados

In [ ]:
# Nombres de los 17 puntos clave
nombres_keypoints = [
    'Nariz', 'Ojo izq', 'Ojo der', 'Oreja izq', 'Oreja der',
    'Hombro izq', 'Hombro der', 'Codo izq', 'Codo der',
    'Muñeca izq', 'Muñeca der', 'Cadera izq', 'Cadera der',
    'Rodilla izq', 'Rodilla der', 'Tobillo izq', 'Tobillo der'
]

print('Puntos detectados por persona:')
print('='*40)
for i, persona in enumerate(results[0].keypoints):
    print(f'\nPersona {i+1}:')
    for j, (x, y, conf) in enumerate(persona.data[0]):
        if conf > 0.5:
            print(f'  {nombres_keypoints[j]}: ({int(x)}, {int(y)}) — confianza: {conf:.2f}')

## Paso 6 — Análisis de postura (ángulo del cuello)

In [ ]:
import numpy as np

def calcular_angulo_vertical(p1, p2):
    """Calcula el ángulo de inclinación respecto al eje vertical."""
    dx = abs(p2[0] - p1[0])
    dy = abs(p2[1] - p1[1])
    if dy == 0:
        return 90
    angulo = np.degrees(np.arctan2(dx, dy))
    return angulo

print('Análisis de postura por persona:')
print('='*40)
for i, persona in enumerate(results[0].keypoints):
    kps = persona.data[0].numpy()

    # Puntos: 3=oreja izq, 5=hombro izq, 11=cadera izq
    oreja  = (kps[3][0], kps[3][1])
    hombro = (kps[5][0], kps[5][1])
    cadera = (kps[11][0], kps[11][1])

    ang_cuello = calcular_angulo_vertical(hombro, oreja)
    ang_torso  = calcular_angulo_vertical(cadera, hombro)

    print(f'\nPersona {i+1}:')
    print(f'  Inclinación cuello: {ang_cuello:.1f}°')
    print(f'  Inclinación torso:  {ang_torso:.1f}°')

    if ang_cuello < 40 and ang_torso < 10:
        print('  ✅ POSTURA CORRECTA')
    else:
        print('  ⚠️ POSTURA INCORRECTA')

## Paso 7 — Comparativa original vs detección

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(img_rgb)
axes[0].set_title('Imagen original')
axes[0].axis('off')
axes[1].imshow(img_resultado_rgb)
axes[1].set_title('Detección YOLO Pose')
axes[1].axis('off')
plt.tight_layout()
plt.show()

---
*IFTS N° 33 — Técnicas de Procesamiento de Imágenes*